## TF-IDF
### Document Frequency (DF)
* DF(t) = Number of documents in the dataset that contain the term t. 
   * ex: If you have 100 documents and the word "insurance" appears in 20 of them: DF(insurance)=20
### Inverse Document Frequency (IDF)
* IDF(t)=log(N/DF(t))
   * Where: N = total number of documents and DF(t) = number of documents containing term t 
   * Example : IDF=log(100/20)=log(5)
### Term Frequency (TF)
* TF(t)= (Number of times term t appears in a document​ / Total number of terms in that document )
   * Example: If a document has 100 words and "insurance" appears 5 times: TF=5/100=0.05

### TF-IDF
* TF-IDF(t)=TF(t)×IDF(t)
* It multiplies:
   * How frequent the word is in that document (TF)
   * How rare the word is across all documents (IDF)
### Intuition 
* TF-IDF answers: How important is this word for this document compared to all other documents?
* High TF-IDF means:
   * The word appears often in this document
   * But does NOT appear in many other 
   * That makes it a good keyword

In [2]:
import pandas as pd
import numpy as np

In [3]:
df = pd.read_csv('/home/ircad/Documents/Learning_text_classification/NLP_datasets/ecommerceDataset.csv')
df.columns = ["label", "text"]
df = df.dropna()
df.head()

,label,text
0,Household,"SAF 'Floral' Framed Painting (Wood, 30 inch x ..."
1,Household,SAF 'UV Textured Modern Art Print Framed' Pain...
2,Household,"SAF Flower Print Framed Painting (Synthetic, 1..."
3,Household,Incredible Gifts India Wooden Happy Birthday U...
4,Household,Pitaara Box Romantic Venice Canvas Painting 6m...


In [4]:
df.label.value_counts()

label
Household                 19312
Books                     11820
Electronics               10621
Clothing & Accessories     8670
Name: count, dtype: int64

In [5]:
df['label_num'] = df.label.map({'Household': 0, 'Books': 1, 'Electronics': 2, 'Clothing & Accessories':3 })
df.head()

,label,text,label_num
0,Household,"SAF 'Floral' Framed Painting (Wood, 30 inch x ...",0
1,Household,SAF 'UV Textured Modern Art Print Framed' Pain...,0
2,Household,"SAF Flower Print Framed Painting (Synthetic, 1...",0
3,Household,Incredible Gifts India Wooden Happy Birthday U...,0
4,Household,Pitaara Box Romantic Venice Canvas Painting 6m...,0


In [6]:
from sklearn.model_selection import train_test_split

X_train, X_test , Y_train, Y_test = train_test_split(df.text, df.label_num, test_size = 0.2, random_state=42, stratify=df.label_num)

print("Shape of X_train: ", X_train.shape)
print("Shape of X_test: ", X_test.shape)

Shape of X_train:  (40338,)
Shape of X_test:  (10085,)


In [7]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report
from sklearn.feature_extraction.text import TfidfVectorizer

clf = Pipeline([('vectorizer_tfidf', TfidfVectorizer()), 
                 ('KNN', KNeighborsClassifier())])

In [8]:
clf.fit(X_train, Y_train)
y_pred  = clf.predict(X_test)
print(classification_report(Y_test, y_pred))


              precision    recall  f1-score   support

           0       0.95      0.96      0.96      3863
           1       0.96      0.95      0.96      2364
           2       0.95      0.95      0.95      2124
           3       0.97      0.97      0.97      1734

    accuracy                           0.96     10085
   macro avg       0.96      0.96      0.96     10085
weighted avg       0.96      0.96      0.96     10085



In [9]:
print(y_pred[:5])
print(Y_test[:5])

[3 0 1 2 0]
38398    3
9667     0
30053    1
45673    2
16663    0
Name: label_num, dtype: int64


In [10]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report
from sklearn.feature_extraction.text import TfidfVectorizer

clf = Pipeline([('vectorizer_tfidf', TfidfVectorizer()), 
                 ('Multi NB', MultinomialNB())])

clf.fit(X_train, Y_train)
y_pred  = clf.predict(X_test)
print(classification_report(Y_test, y_pred))

              precision    recall  f1-score   support

           0       0.90      0.98      0.94      3863
           1       0.97      0.92      0.95      2364
           2       0.97      0.90      0.93      2124
           3       0.98      0.94      0.96      1734

    accuracy                           0.94     10085
   macro avg       0.95      0.94      0.94     10085
weighted avg       0.94      0.94      0.94     10085



In [ ]:
from sklearn.ensemble import RandomForestClassifier

clf = Pipeline([('vectorizer_tfidf', TfidfVectorizer()), 
                 ('Random Forest', RandomForestClassifier())])

clf.fit(X_train, Y_train)
y_pred  = clf.predict(X_test)
print(classification_report(Y_test, y_pred))

In [ ]:
import spacy

nlp = spacy.load('en_core_web_sm')
filtered_tokens = []

def preprocess(text):
    doc = nlp(text)
    for token in doc:
        if token.is_stop or token.is_punct:
            continue
        filtered_tokens.append(token.lemma_)

    return " ".join(filtered_tokens)

In [ ]:
df['preprocess_text'] = df['text'].apply(preprocess)
df.head()

In [ ]:
print(df.text[0])

SAF 'Floral' Framed Painting (Wood, 30 inch x 10 inch, Special Effect UV Print Textured, SAO297) Painting made up in synthetic frame with UV textured print which gives multi effects and attracts towards it. This is an special series of paintings which makes your wall very beautiful and gives a royal touch (A perfect gift for your special ones).


In [ ]:
print(df.preprocess_text[0])

In [ ]:
X_train, X_test , Y_train, Y_test = train_test_split(df.preprocess_text, df.label_num, test_size = 0.2, random_state=42, stratify=df.label_num)

In [ ]:
from sklearn.ensemble import RandomForestClassifier

clf = Pipeline([('vectorizer_tfidf', TfidfVectorizer()), 
                 ('Random Forest', RandomForestClassifier())])

clf.fit(X_train, Y_train)
y_pred  = clf.predict(X_test)
print(classification_report(Y_test, y_pred))